# Semantic Router Evaluation Pipeline

This notebook evaluates the `SemanticRouterModule` performance. It classifies legal queries into **Reasoning-LLM**, **General-LLM**, or **Casual-LLM** routes.

**Current Task:** Benchmarking between Qwen-2.5 and Gemini-2.5 Flash Lite as requested by the team leader.

In [1]:
# 1. Install Dependencies
%pip install pandas tqdm requests python-dotenv rich -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys, os, time
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

# 1. Path Management
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.append(project_root)

# 2. Load Environment (MUST happen before Framework imports)
dotenv_path = os.path.join(project_root, '.env')
if load_dotenv(dotenv_path):
    print(f"Environment loaded from: {dotenv_path}")
else:
    print("WARNING: .env file not found or could not be loaded.")

# 3. Framework Imports
from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.modules.router import SemanticRouterModule
from src.adaptive_routing.config import FrameworkConfig

print("Framework components imported successfully.")

c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment loaded from: c:\Users\Prince\Documents\GitHub\Legal-Adaptive-Routing-Framework\.env
Framework components imported successfully.


In [3]:
# ==================================================================
# 1. CONFIGURATION & HYPERPARAMETERS (LEADER'S REQUEST)
# ==================================================================

# --- API Credentials ---
OPENROUTER_API_KEY = ""  # Leave empty to use .env file
QUERY_LIMIT = 5  # @param {type:"integer"} # Set to None for full evaluation

# --- Model Options (Choose one) ---
MODEL_OPTIONS = {
    "Qwen-2.5": "qwen/qwen-2.5-7b-instruct",
    "Gemini-2.5": "google/gemini-2.5-flash-lite"
}

SELECTED_MODEL = "Qwen-2.5" # @param ["Qwen-2.5", "Gemini-2.5"]

CONFIG = {
    "api_key": OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY"),
    "triage_model": MODEL_OPTIONS[SELECTED_MODEL], 
    "router_model": MODEL_OPTIONS[SELECTED_MODEL],
    "router_temp": 0.1,                   
    "router_max_tokens": 250,         
    "router_use_system": True,    
    "router_reasoning": False              
}

SYSTEM_PROMPT = "" # Leave empty to use framework defaults

# ==================================================================
# 2. INITIALIZATION & OVERRIDES
# ==================================================================

# Apply API Key override if provided
if OPENROUTER_API_KEY: os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("CRITICAL ERROR: OPENROUTER_API_KEY not found in .env or config.")
else:
    print(f"API Key detected (Length: {len(api_key)})")

try:
    # Force update FrameworkConfig with the API Key and Model settings
    FrameworkConfig._update_settings_(**CONFIG)
    print(f"Framework settings updated for {SELECTED_MODEL}.")
except Exception as e:
    print(f"Configuration Error: {e}")

triage = TriageModule(api_key=api_key)
router = SemanticRouterModule(api_key=api_key)

if SYSTEM_PROMPT.strip():
    router._classifier._system_prompt = SYSTEM_PROMPT
    print("Custom system prompt applied to Router.")

print(f"Engine ready with model: {FrameworkConfig._ROUTER_MODEL}")

API Key detected (Length: 73)
Framework settings updated for Qwen-2.5.
Engine ready with model: qwen/qwen-2.5-7b-instruct


In [4]:
# ==================================================================
# 3. QUICK TEST (Single Inference)
# ==================================================================
sample_query = "How do I file a case for illegal dismissal?"

print(f"Testing: '{sample_query}'")
try:
    start_time = time.time()
    triage_res = triage._process_request_(sample_query)
    norm_text = triage_res.get("normalized_text", sample_query)
    
    route_res = router._process_routing_(norm_text)
    duration = time.time() - start_time
    
    print("\n--- RESULTS ---")
    print(f"Detected Route: {route_res.get('route')}")
    print(f"Confidence: {route_res.get('confidence')}")
    print(f"Latency: {duration:.2f} seconds")
except Exception as e:
    print(f"Error: {e}")

Testing: 'How do I file a case for illegal dismissal?'

--- RESULTS ---
Detected Route: General-LLM
Confidence: 0.95
Latency: 7.53 seconds


In [5]:
# 4. Load Evaluation Dataset
dataset_path = 'dataset/Routing-Evaluation-Dataset.csv'
df = pd.read_csv(dataset_path)

if QUERY_LIMIT is not None:
    df = df.head(QUERY_LIMIT).copy()

input_column = next((c for c in ["Query", "Prompt", "prompts"] if c in df.columns), df.columns[0])
print(f"Processing dataset: {len(df)} rows | Input column: '{input_column}'")
df.head()

Processing dataset: 5 rows | Input column: 'Query'


,Query,Expected,Groq Expected,Groq Reason,Predicted
0,Tatlong OFW na nasa deathrow sa China... Naha...,Reasoning-LLM,Reasoning-LLM,The query involves a specific scenario of OFWs...,NaN
1,Ang Passport and Employment Contract ko ay nas...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation with p...,NaN
2,pag may pandemic. maiiwan ba ako here sa hongk...,General-LLM,Reasoning-LLM,The query involves a personal situation of an ...,NaN
3,Paano gumagana yung bagong Senate Bill No. 2390?,General-LLM,General-LLM,The query asks for general information about t...,NaN
4,Nawalan ako ng trabaho dahil I made a Case sa ...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation and as...,NaN


In [6]:
# 5. Execute Evaluation
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

print(f"Starting evaluation processing for {SELECTED_MODEL}...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    if pd.notna(row.get(pred_col)) and row.get(pred_col) != "": continue
    
    try:
        query = str(row[input_column])
        t_res = triage._process_request_(query)
        norm_text = t_res.get("normalized_text", query)
        r_res = router._process_routing_(norm_text)
        
        df.at[index, pred_col] = r_res.get('route')
        
        if (index + 1) % 10 == 0: df.to_csv(checkpoint_path, index=False)
        time.sleep(0.5)
    except Exception as e:
        print(f"Error on row {index}: {e}")
        df.at[index, pred_col] = f"ERROR: {str(e)[:50]}"

print("Evaluation cycle complete.")

Starting evaluation processing for Qwen-2.5...


 80%|████████  | 4/5 [00:38<00:07,  7.64s/it]Language tag not found in normalizer output. First 100 chars: You lost your job because you made a case of sexual assault against your boss. What can you do?

 De
100%|██████████| 5/5 [00:43<00:00,  8.73s/it]

Evaluation cycle complete.


In [7]:
# 6. Summary & Results
output_path = f'dataset/Routing-Evaluation-Results-{SELECTED_MODEL}.csv'
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

if 'Expected' in df.columns:
    eval_df = df[df[pred_col].notna() & (~df[pred_col].str.startswith("ERROR"))]
    if not eval_df.empty:
        accuracy = (eval_df[pred_col] == eval_df['Expected']).mean() * 100
        print(f"\n{SELECTED_MODEL} Accuracy: {accuracy:.2f}%")
        print("\nDistribution:")
        print(eval_df[pred_col].value_counts())
    else:
        print("No valid predictions to evaluate.")
df.head()

Results saved to dataset/Routing-Evaluation-Results-Qwen-2.5.csv

Qwen-2.5 Accuracy: 80.00%

Distribution:
Predicted_Qwen-2.5
Reasoning-LLM    4
General-LLM      1
Name: count, dtype: int64


,Query,Expected,Groq Expected,Groq Reason,Predicted,Predicted_Qwen-2.5
0,Tatlong OFW na nasa deathrow sa China... Naha...,Reasoning-LLM,Reasoning-LLM,The query involves a specific scenario of OFWs...,NaN,Reasoning-LLM
1,Ang Passport and Employment Contract ko ay nas...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation with p...,NaN,Reasoning-LLM
2,pag may pandemic. maiiwan ba ako here sa hongk...,General-LLM,Reasoning-LLM,The query involves a personal situation of an ...,NaN,Reasoning-LLM
3,Paano gumagana yung bagong Senate Bill No. 2390?,General-LLM,General-LLM,The query asks for general information about t...,NaN,General-LLM
4,Nawalan ako ng trabaho dahil I made a Case sa ...,Reasoning-LLM,Reasoning-LLM,The query involves a personal situation and as...,NaN,Reasoning-LLM
